<a href="https://colab.research.google.com/github/Sujal-Sharma-Codes/Prodigy-GA-4/blob/main/pix2pix_image_translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task-04: Image-to-Image Translation with Pix2Pix (cGAN)
Implement Pix2Pix — a conditional GAN that translates an input image into a corresponding output image (e.g. building facade labels -> real facade photos).

**Before running:** Runtime -> Change runtime type -> T4 GPU.

## 1. Install libraries

In [8]:
!pip -q install torch torchvision

## 2. Download the dataset
Using the CMP Facades dataset (paired label/photo images, ~30 MB), the same dataset used in the original Pix2Pix paper. Each image in the dataset is a single file with the two paired images side by side.

In [9]:
import os, urllib.request, tarfile

url = "http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/facades.tar.gz"
tar_path = "facades.tar.gz"

if not os.path.exists("facades"):
    urllib.request.urlretrieve(url, tar_path)
    with tarfile.open(tar_path) as t:
        t.extractall(".")

print(os.listdir("facades"))
print("Train images:", len(os.listdir("facades/train")))

['test', 'train', 'val']
Train images: 400


## 3. Dataset class
Each file is a side-by-side pair: left half = target photo, right half = input label map (this is the standard layout for this dataset).

In [10]:
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

IMG_SIZE = 256

transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.5] * 3, [0.5] * 3),
])

class FacadesDataset(Dataset):
    def __init__(self, root):
        self.files = sorted(glob.glob(os.path.join(root, "*.jpg")))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        w, h = img.size
        photo = img.crop((0, 0, w // 2, h))       # target (real photo)
        label = img.crop((w // 2, 0, w, h))       # input (label map)
        return transform(label), transform(photo)  # (input, target)

train_ds = FacadesDataset("facades/train")
val_ds = FacadesDataset("facades/val")
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=True)

print("Train pairs:", len(train_ds), "| Val pairs:", len(val_ds))

Train pairs: 400 | Val pairs: 100


## 4. Generator: U-Net
An encoder-decoder with skip connections, so fine details from the input are passed directly to the output.

In [11]:
import torch.nn as nn

class UNetDown(nn.Module):
    def __init__(self, in_c, out_c, normalize=True, dropout=0.0):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=False)]
        if normalize:
            layers.append(nn.InstanceNorm2d(out_c))
        layers.append(nn.LeakyReLU(0.2))
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_c, out_c, dropout=0.0):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=False),
                  nn.InstanceNorm2d(out_c), nn.ReLU()]
        if dropout:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)
    def forward(self, x, skip):
        x = self.model(x)
        return torch.cat([x, skip], dim=1)

class GeneratorUNet(nn.Module):
    def __init__(self, in_c=3, out_c=3):
        super().__init__()
        self.d1 = UNetDown(in_c, 64, normalize=False)
        self.d2 = UNetDown(64, 128)
        self.d3 = UNetDown(128, 256)
        self.d4 = UNetDown(256, 512, dropout=0.5)
        self.d5 = UNetDown(512, 512, dropout=0.5)
        self.d6 = UNetDown(512, 512, dropout=0.5)
        self.d7 = UNetDown(512, 512, dropout=0.5)
        self.d8 = UNetDown(512, 512, normalize=False, dropout=0.5)

        self.u1 = UNetUp(512, 512, dropout=0.5)
        self.u2 = UNetUp(1024, 512, dropout=0.5)
        self.u3 = UNetUp(1024, 512, dropout=0.5)
        self.u4 = UNetUp(1024, 512, dropout=0.5)
        self.u5 = UNetUp(1024, 256)
        self.u6 = UNetUp(512, 128)
        self.u7 = UNetUp(256, 64)
        self.final = nn.Sequential(
            nn.ConvTranspose2d(128, out_c, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, x):
        d1 = self.d1(x); d2 = self.d2(d1); d3 = self.d3(d2); d4 = self.d4(d3)
        d5 = self.d5(d4); d6 = self.d6(d5); d7 = self.d7(d6); d8 = self.d8(d7)
        u1 = self.u1(d8, d7); u2 = self.u2(u1, d6); u3 = self.u3(u2, d5)
        u4 = self.u4(u3, d4); u5 = self.u5(u4, d3); u6 = self.u6(u5, d2)
        u7 = self.u7(u6, d1)
        return self.final(u7)

## 5. Discriminator: PatchGAN
Instead of classifying the whole image as real/fake in one shot, PatchGAN classifies overlapping 70x70 patches. This pushes the generator to get local texture and detail right.

In [12]:
class Discriminator(nn.Module):
    def __init__(self, in_c=3):
        super().__init__()
        def block(ic, oc, normalize=True):
            layers = [nn.Conv2d(ic, oc, 4, 2, 1)]
            if normalize:
                layers.append(nn.InstanceNorm2d(oc))
            layers.append(nn.LeakyReLU(0.2))
            return layers
        self.model = nn.Sequential(
            *block(in_c * 2, 64, normalize=False),
            *block(64, 128),
            *block(128, 256),
            *block(256, 512),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(512, 1, 4, padding=1, bias=False),
        )

    def forward(self, img_input, img_target):
        # discriminator is conditioned on the input image (the "c" in cGAN)
        x = torch.cat([img_input, img_target], dim=1)
        return self.model(x)

## 6. Losses, optimizers, setup
Pix2Pix combines two losses for the generator:
- **Adversarial loss:** fool the discriminator (standard GAN loss)
- **L1 loss:** stay close to the real target image, weighted by `lambda_l1` (this is what keeps colors/structure correct, not just "realistic-looking")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

G_model = GeneratorUNet().to(device)
D_model = Discriminator().to(device)

criterion_gan = nn.MSELoss()   # LSGAN-style loss, more stable than BCE
criterion_l1 = nn.L1Loss()
lambda_l1 = 100

opt_G = torch.optim.Adam(G_model.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(D_model.parameters(), lr=2e-4, betas=(0.5, 0.999))

patch = (1, IMG_SIZE // 16 - 1, IMG_SIZE // 16 - 1)  # PatchGAN output shape

## 7. Training loop
For a class demo, 20 epochs on the ~400 training pairs is enough to see clear translation, in roughly 15-25 minutes on a T4. Increase `n_epochs` for sharper results.

In [ ]:
import time

n_epochs = 20

def sample_and_show(epoch):
    G_model.eval()
    inp, tgt = next(iter(val_loader))
    inp, tgt = inp.to(device), tgt.to(device)
    with torch.no_grad():
        fake = G_model(inp)
    grid = torch.cat([inp[:3], fake[:3], tgt[:3]], dim=0)
    grid = (grid * 0.5 + 0.5).clamp(0, 1)  # unnormalize
    from torchvision.utils import make_grid, save_image
    save_image(make_grid(grid, nrow=3), f"outputs/epoch_{epoch}.png")
    G_model.train()

os.makedirs("outputs", exist_ok=True)

for epoch in range(1, n_epochs + 1):
    start = time.time()
    for inp, tgt in train_loader:
        inp, tgt = inp.to(device), tgt.to(device)
        valid = torch.ones((inp.size(0), *patch), device=device)
        fake_lbl = torch.zeros((inp.size(0), *patch), device=device)

        # ---- train generator ----
        opt_G.zero_grad()
        fake = G_model(inp)
        pred_fake = D_model(inp, fake)
        loss_gan = criterion_gan(pred_fake, valid)
        loss_l1 = criterion_l1(fake, tgt)
        loss_G = loss_gan + lambda_l1 * loss_l1
        loss_G.backward()
        opt_G.step()

        # ---- train discriminator ----
        opt_D.zero_grad()
        pred_real = D_model(inp, tgt)
        loss_real = criterion_gan(pred_real, valid)
        pred_fake = D_model(inp, fake.detach())
        loss_fake = criterion_gan(pred_fake, fake_lbl)
        loss_D = 0.5 * (loss_real + loss_fake)
        loss_D.backward()
        opt_D.step()

    print(f"Epoch {epoch}/{n_epochs}  G_loss={loss_G.item():.3f}  D_loss={loss_D.item():.3f}  "
          f"({time.time()-start:.0f}s)")
    if epoch % 5 == 0 or epoch == n_epochs:
        sample_and_show(epoch)

## 8. Compare results across training
Each saved grid shows, left to right: **input label map -> generated photo -> real photo**.

In [13]:
from IPython.display import display, Image as IPImage
import glob as g

for f in sorted(g.glob("outputs/epoch_*.png"), key=lambda p: int(p.split("_")[-1].split(".")[0])):
    print(f)
    display(IPImage(f))

## 9. Save the trained generator

In [14]:
torch.save(G_model.state_dict(), "pix2pix_generator.pth")
print("Saved pix2pix_generator.pth")

Saved pix2pix_generator.pth


## 10. Notes
- **Dataset:** CMP Facades (400 train / 100 val paired images of building facade label maps and real photos), from the original Pix2Pix paper.
- To translate a different kind of image pair (e.g. sketches -> photos, maps -> satellite), swap the dataset for your own paired images and keep the same input/target split logic.
- Pix2Pix needs **paired** data (input and target must correspond exactly). For translation between unpaired image sets, CycleGAN is the standard alternative.